# 18 · E2d: how slow may the hardware's own dynamics be? Timescale separation as a condition for software

**Where E2–E2c left off.** The fitted L1 survives every realization so far:
* other numerical integrators (Notebook 05);
* spiking neurons, at ~100 per unit (Notebook 16);
* conductance-based synapses, once each neuron's leak is retuned (Notebook 17: gap 0.002 × noise, ΔFEVE −0.005).

In each case the substrate's *extra* dynamics were either **fast** (spikes, τ_m = 20 ms) or **exactly compensated at rest**.

**Real neurons also carry slow voltage-gated currents**, e.g. K⁺ conductances that activate over hundreds of milliseconds to seconds. Such a current belongs to the hardware, not to the fitted program. A program is "software" only if what it computes does not depend on such details. That classically requires **timescale separation**: the hardware's own dynamics must be much faster (or much slower) than the computation's.

**Realization.** A fraction ρ of each neuron's leak is carried by a slow K⁺-like conductance with gating time constant τ_n, linearised around rest:

  τ_i dV_i/dt = −(1 − ρ) V_i − ρ w_i + … (the rest of L1 unchanged)  τ_n dw_i/dt = −w_i + V_i

* At steady state w = V, so the **fixed point and the DC gain are exactly L1's**, and the realization tends to L1 as τ_n → 0.
* For slow τ_n the membrane responds first with less leak (a larger transient) and then adapts. The substrate adds an *adaptation* the fitted program does not have: the ladder's L2 mechanism, placed in the hardware.

It runs on two synapse bases: L1's `current` synapses, and E2c's leak-compensated `conductance` synapses at the physiological scale. The second is the most biophysical realization so far.

**What the synthetic check showed while building this** (a known L1, 30 neurons; its fitted time constants τ ≈ 0.5–0.6 s), at ρ = 0.3 on current synapses:

| τ_n | 0.05 s | 0.2 s | 1 s | 5 s | 20 s |
|---|---|---|---|---|---|
| gap/noise | 0.004 | 0.08 | **3.5** | **21** | **16** |

* The program survives only while the slow current is **faster than the program's own time constants**.
* The damage is not monotone in τ_n: a 20-s current hardly moves within the 20-s response window, so it changes the response less than a 5-s one.

In [ ]:
#@title Setup — run first (works locally on a Mac/PC and on Colab)
import os, sys, subprocess
REPO_URL   = "https://github.com/aslansd/Closure-Bench.git"  # Colab only: clone from here
DRIVE_PATH = "/content/drive/MyDrive/Closure-Bench"          # Colab only: ...or use a Drive copy
IN_COLAB = "google.colab" in sys.modules

def _find_package():
    # 1) explicit location, 2) the repo this notebook lives in, 3) Colab clone / Drive
    cands = [os.environ.get("CLOSUREBENCH_ROOT", ""), ".", "..", "/content/Closure-Bench", DRIVE_PATH]
    for root in cands:
        if root and os.path.isdir(os.path.join(root, "closurebench")):
            return os.path.abspath(root)
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wormneuroatlas", "optax"], check=False)
        if subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Closure-Bench"]).returncode == 0:
            return "/content/Closure-Bench"
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(os.path.join(DRIVE_PATH, "closurebench")):
            return DRIVE_PATH
    raise RuntimeError("closurebench not found. Locally: open this notebook from Closure-Bench/notebooks/, "
                       "or set the environment variable CLOSUREBENCH_ROOT to the Closure-Bench folder.")

ROOT = _find_package()
sys.path.insert(0, ROOT)
try:
    import jax, optax, wormneuroatlas  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Locally, create the environment first: see README.md "
                      "('Run locally'), then select the 'Python (closurebench)' kernel.") from None
RESULTS = os.environ.get("CLOSUREBENCH_RESULTS", os.path.join(ROOT, "results"))
os.makedirs(RESULTS, exist_ok=True)
from closurebench import config as CBC
MODE = CBC.get_mode()        # 'smoke' | 'laptop' | 'full'  (env CLOSUREBENCH_MODE, default: full on GPU, laptop on CPU)
print("closurebench:", ROOT); print("results:", RESULTS); print(CBC.describe_machine()); print("MODE:", MODE)

In [ ]:
import json, glob, pickle, time, datetime, numpy as np, matplotlib.pyplot as plt, jax
import closurebench
assert tuple(int(x) for x in closurebench.__version__.split(".")[:3]) >= (0, 16, 0), \
    f"Notebook 18 needs closurebench >= 0.16.0 (found {closurebench.__version__}); copy the full package and restart the kernel"
from closurebench import data as D, ladder as lad, metrics as M_, substrates as SB, conductance as CD, intrinsic as IN
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
CONF = {"smoke":  dict(N_COLS=10, FOLDS_USED=1),
        "laptop": dict(N_COLS=None, FOLDS_USED=None),
        "full":   dict(N_COLS=None, FOLDS_USED=None)}[MODE]
globals().update(CONF)
RHOS, TAU_NS, RHO_PRIMARY = (0.1, 0.3, 0.6), (0.05, 0.2, 1.0, 5.0, 20.0), 0.3
BASES = ("current", "conductance")
GAP_MAX, DFEVE_MAX = 0.1, 0.05
V_LO, V_HI, E_EXC, E_INH = -70.0, -20.0, 0.0, -80.0
TAG = "" if MODE == "full" else f"_{MODE}"

# ---- the fitted L1 models of Notebook 04 (as in Notebook 05); smoke falls back to a synthetic L1 if they are absent
E1_TAG = "_laptop" if MODE == "smoke" else TAG
try:
    e1_result = json.load(open(os.path.join(RESULTS, f"E1_result{E1_TAG}.json")))
    e1_plan = json.load(open(os.path.join(RESULTS, e1_result["preregistration"])))
    assert e1_result["L_star"] == "L1", "conductance.py implements L1"
    c = e1_plan["config"]
    ds = D.Dataset.load(os.path.join(RESULTS, "atlas_dataset.npz"))
    if c["n_neurons"] < ds.N:
        mk = ds.mask()[0]; score = mk.sum(1).astype(float); score[ds.stim] += mk.sum(0)
        ds = D.subset(ds, np.sort(np.argsort(-score)[:c["n_neurons"]]))
    st = lad.Structure.from_dataset(ds, tie=c["tie_classes"]); cfg = lad.SimConfig(**c["sim"])
    folds = M_.kfold_pairs(ds, c["k_folds"], c["fold_seed"])
    ck1 = os.path.join(RESULTS, f"e1_checkpoints{E1_TAG}")
    models = [pickle.load(open(os.path.join(ck1, f"fold{f}_L1_final.pkl"), "rb"))["params"] for f in range(c["k_folds"])]
    SOURCE = f"E1 fits ({E1_TAG.strip('_') or 'full'})"
except (FileNotFoundError, KeyError) as err:
    if MODE != "smoke":
        raise
    print("E1 fits not found (", err, ") -> smoke test on a synthetic L1")
    ds, p_true, _ = lad.make_synthetic(lad.random_wiring(N=30, S=10, seed=1, strains=("wt",)), true_level="L1", seed=1)
    st = lad.Structure.from_dataset(ds); cfg = lad.SimConfig(); folds = [None]; models = [p_true]; SOURCE = "synthetic L1"
rng = np.random.default_rng(0)
COLS = np.sort(rng.choice(ds.S, N_COLS, replace=False)) if N_COLS and N_COLS < ds.S else np.arange(ds.S)
FOLDS = list(range(len(models)))[:FOLDS_USED] if FOLDS_USED else list(range(len(models)))
print(f"source: {SOURCE} | {ds.N} neurons | {len(COLS)} stimuli | fold models {FOLDS}")
print(f"slow-current share rho {RHOS} (primary {RHO_PRIMARY}); time constants tau_n {TAU_NS} s; synapse bases {BASES}")

### Pre-registration (before §1)

Recorded in `results/E2d_preregistration*.json`. The reference and the pass criteria are as in Notebooks 05, 16 and 17: `ode` (RK4 dt = 0.02 s); gap/noise < 0.1 **and** |ΔFEVE| < 0.05 on held-out pairs, for every fold model.

The primary condition is ρ = 0.3 on the `conductance` base.

* **TIMESCALE_SEPARATION:** the realization passes at τ_n = 0.05 s **and** fails at some τ_n ≥ 1 s. The fitted program is then software only if the hardware's slow currents are faster than the program's own time constants.
* **INVISIBLE_HARDWARE:** it passes at *every* τ_n. Slow intrinsic currents of this size would then be invisible to the atlas, the same blind spot that made L2 unidentifiable in E1 (Notebooks 02, 07, 09).
* Descriptive:
  * τ\*(ρ, base), the largest passing τ_n;
  * its ratio to the fitted model's median membrane time constant.

In [ ]:
prereg = os.path.join(RESULTS, f"E2d_preregistration{TAG}.json")
plan = {"experiment": "E2d slow intrinsic currents", "mode": MODE, "registered_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "source": SOURCE, "config": {"N_COLS": int(len(COLS)), "folds": FOLDS, "rho": list(RHOS), "tau_n_s": list(TAU_NS), "bases": list(BASES),
                                     "primary": {"rho": RHO_PRIMARY, "base": "conductance"}, "v_range_mV": [V_LO, V_HI], "E_exc": E_EXC, "E_inh": E_INH},
        "pass": f"gap/noise < {GAP_MAX} and |dFEVE| < {DFEVE_MAX} vs ode, every fold model; unstable = fail",
        "TIMESCALE_SEPARATION": "primary condition passes at tau_n = 0.05 s and fails at some tau_n >= 1 s",
        "INVISIBLE_HARDWARE": "primary condition passes at every tau_n"}
if os.path.exists(prereg):
    plan = json.load(open(prereg)); print("Existing registration (", plan["registered_at"], ") used unchanged.")
else:
    json.dump(plan, open(prereg, "w"), indent=2); print("Registered ->", prereg)

## 1 · Every fold model × base × ρ × τ_n

Seconds per run; ~15–20 minutes in all on a laptop.

In [ ]:
rows, TAU_FIT = [], []
for f in FOLDS:
    p = jax.tree_util.tree_map(jax.numpy.asarray, models[f])
    e = SB.effective_params("L1", p, st, COLS)
    R_ode = SB.simulate_numpy(e, ds.stim[COLS], cfg, "rk4", dt=0.02)
    Vr, Sr = CD.rest_state(e, cfg)
    syn = CD.compile_synapses(e, Vr, Sr, CD.voltage_map(Vr, V_LO, V_HI), scale=1.0, E_exc=E_EXC, E_inh=E_INH)
    tm = None if folds[f] is None else folds[f]
    TAU_FIT.append(float(np.median(e["tau"])))
    print(f"fold {f}: fitted membrane time constants median {np.median(e['tau']):.2f} s (range {e['tau'].min():.2f}-{e['tau'].max():.2f}), synaptic {e['tau_s']:.2f} s")
    for base in BASES:
        for rho in RHOS:
            line = []
            for tn in TAU_NS:
                Rs = IN.simulate_slow_k(e, ds.stim[COLS], cfg, Vr, rho=rho, tau_n=tn, syn=syn if base == "conductance" else None)
                ok = bool(np.all(np.isfinite(Rs)))
                r = dict(fold=f, base=base, rho=rho, tau_n=tn, stable=ok,
                         gap=SB.substrate_gap(Rs, R_ode, ds, COLS) if ok else float("inf"),
                         dfeve=(SB.feve_on(Rs, ds, COLS, tm) - SB.feve_on(R_ode, ds, COLS, tm)) if ok else float("-inf"))
                rows.append(r); line.append(f"{tn:g}s {r['gap']:.3f}/{r['dfeve']:+.3f}" if ok else f"{tn:g}s UNSTABLE")
            print(f"  {base:11s} rho {rho}: " + " | ".join(line), flush=True)

## 2 · How slow may the hardware be?

In [ ]:
passes = lambda r: r["stable"] and r["gap"] < GAP_MAX and abs(r["dfeve"]) < DFEVE_MAX
def all_pass(base, rho, tn):
    rs = [r for r in rows if r["base"] == base and r["rho"] == rho and r["tau_n"] == tn]
    return bool(rs) and all(passes(r) for r in rs)
TSTAR = {(b, rho): max([tn for tn in TAU_NS if all_pass(b, rho, tn)], default=None) for b in BASES for rho in RHOS}
tau_fit = float(np.median(TAU_FIT))
print(f"fitted membrane time constant (median over folds): {tau_fit:.2f} s\n")
print(f"{'base':11s} {'rho':>4s} | " + " ".join(f"{tn:>9g}s" for tn in TAU_NS) + " | tau*      (cells: mean gap/noise over folds; * = all folds pass)")
for b in BASES:
    for rho in RHOS:
        cells = []
        for tn in TAU_NS:
            g = np.mean([r["gap"] for r in rows if r["base"] == b and r["rho"] == rho and r["tau_n"] == tn])
            cells.append(f"{g:9.3f}{'*' if all_pass(b, rho, tn) else ' '}")
        ts = TSTAR[(b, rho)]
        print(f"{b:11s} {rho:4.1f} | " + " ".join(cells) + f" | {ts if ts is not None else '< ' + str(TAU_NS[0])}" + (f" s ({ts / tau_fit:.2f} x tau_fit)" if ts else ""))
prim = ("conductance", RHO_PRIMARY)
TIMESCALE_SEPARATION = all_pass(*prim, TAU_NS[0]) and any(not all_pass(*prim, tn) for tn in TAU_NS if tn >= 1.0)
INVISIBLE_HARDWARE = all(all_pass(*prim, tn) for tn in TAU_NS)
print("\nRegistered rules -> TIMESCALE_SEPARATION:", TIMESCALE_SEPARATION, "| INVISIBLE_HARDWARE:", INVISIBLE_HARDWARE)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for a, b in zip(ax, BASES):
    for rho, c in zip(RHOS, ("tab:green", "tab:blue", "tab:red")):
        g = [np.mean([min(r["gap"], 1e4) for r in rows if r["base"] == b and r["rho"] == rho and r["tau_n"] == tn]) for tn in TAU_NS]
        a.plot(TAU_NS, g, "o-", c=c, label=f"ρ = {rho}")
    a.axhline(GAP_MAX, c="k", ls="--", lw=0.6); a.axvline(tau_fit, c="0.5", ls=":", lw=0.8)
    a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel("slow-current time constant τ_n (s)"); a.set_title(f"{b} synapses")
    a.legend(frameon=False, fontsize=8)
ax[0].set_ylabel("gap / noise vs ode (mean over folds)")
plt.tight_layout(); plt.show()

In [ ]:
out = {"source": SOURCE, "timescale_separation": TIMESCALE_SEPARATION, "invisible_hardware": INVISIBLE_HARDWARE,
       "tau_star": {f"{b}|{rho}": v for (b, rho), v in TSTAR.items()}, "tau_fit_median": tau_fit, "rows": rows}
json.dump(out, open(os.path.join(RESULTS, f"E2d_result{TAG}.json"), "w"), indent=2, default=float)
print("saved", f"E2d_result{TAG}.json")

## 3 · Reading the result

| TIMESCALE_SEPARATION | INVISIBLE_HARDWARE | reading for C2 |
|---|---|---|
| yes | no | The fitted program is software **under timescale separation**: it survives any realization whose own slow dynamics are faster than its time constants (τ\* above), and changes measurably when the hardware has slower currents. A biological realization must keep its slow conductances fast, small, or compensated. This is the classical condition for a level of description to close. |
| no | yes | Slow intrinsic currents of this size are **invisible** to the atlas: hardware adaptation hides under the fitted program. This is the substrate-side face of E1's finding that L2 is unidentifiable. The data cannot tell a program with adaptation from a program on adapting hardware. |
| no | no | The program fails even with fast hardware currents. With the conductance base this would point to the compensated compiler's fragility; compare the `current` base. |

**What closes here.** E2 (Notebooks 05, 16, 17, 18) asked whether the fitted level is independent of its realization:
* across integrators: yes;
* across spiking equations: yes, with ~100 neurons per unit;
* across conductance synapses: yes, if the leak is retuned;
* across slow intrinsic currents: this notebook's answer.

**Limits.**
* The slow current is linearised and has one time constant for all neurons. Real channels are nonlinear and heterogeneous.
* The window-mean read-out (20 s) is itself slow. A time-resolved read-out (Notebook 06) would see faster hardware dynamics, so τ\* is an upper bound for what time courses would tolerate.